# R5 — Grounded RAG chain

One function an adjuster can call with a policy question: `answer_policy_question(q)` returns `{'answer': ..., 'cited_chunk_ids': [...]}` — the answer comes ONLY from the retrieved policy sections (vector search top-5 on `claims.genai.policy_chunks_idx`), and when the binder has no answer it replies with exactly **`not in policy`**.

Cells:
1. The chain: retrieval + grounded LLM call (endpoint `databricks-meta-llama-3-3-70b-instruct`).
2. Run the whole gold set (G01–G19, G19 with its structured claim facts from `claims.gold.claims_enriched`) plus the two unanswerables U01/U02, and rebuild `claims.genai.gold_answers` (question_id, question, answer, cited_chunk_ids ARRAY<STRING>; one row per question).
3. Verification: U01/U02 say `not in policy`; >=16 gold questions answered with no phrase leak; citations 1–5 in rank order.
4. Final `gold_answers` listing.

In [0]:
%python
# Step 1 — the governed RAG chain.
# retrieve: vector search top-5 on the TRIGGERED Delta Sync index claims.genai.policy_chunks_idx
# generate: grounded system prompt; the ONLY fallback allowed is the exact phrase 'not in policy'

import re, requests
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_HOST = _w.config.host if _w.config.host.startswith("http") else "https://" + _w.config.host
_AUTH = _w.config.authenticate()["Authorization"]
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
NOT_IN_POLICY = "not in policy"


def _chat(endpoint, system, user, max_tokens=600):
    """One chat-completion call to a model-serving endpoint."""
    r = requests.post(
        f"{_HOST}/serving-endpoints/{endpoint}/invocations",
        headers={"Authorization": _AUTH, "Content-Type": "application/json"},
        json={"messages": [{"role": "system", "content": system},
                           {"role": "user", "content": user}],
              "max_tokens": max_tokens, "temperature": 0.0},
        timeout=180,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def _retrieve(question, k=5):
    """Top-k chunk ids from the policy vector index, best first (rank order)."""
    qlit = question.replace("'", "''")
    df = spark.sql(f"""
      SELECT vs.chunk_id
      FROM (SELECT '{qlit}' AS qtxt) g,
      LATERAL vector_search(index => 'claims.genai.policy_chunks_idx',
                            query_text => g.qtxt, num_results => {k}) vs
    """)
    return [row.chunk_id for row in df.collect()]


_chunk_cache = {}
def _chunk_text(chunk_id):
    if chunk_id not in _chunk_cache:
        _chunk_cache[chunk_id] = spark.sql(
            "SELECT text FROM claims.genai.policy_chunks WHERE chunk_id = '{}'".format(
                chunk_id.replace("'", "''"))
        ).collect()[0][0]
    return _chunk_cache[chunk_id]


SYSTEM_PROMPT = (
    "You are the Saffron Shield policy binder assistant for insurance adjusters. "
    "Answer the question using ONLY the policy sections provided in the context. "
    "If the context does not contain the information needed to answer, reply with exactly: "
    "not in policy\n"
    "and nothing else. Never use the phrase 'not in policy' unless the context truly cannot "
    "answer the question. Never use outside knowledge. Be concise and factual; quote the "
    "relevant rule from the sections."
)


def answer_policy_question(question, extra_context=None, k=5):
    """Return {'answer': str, 'cited_chunk_ids': [chunk_id, ...]} (1..k, best first).

    The answer is generated ONLY from the retrieved chunk texts (plus optional
    structured facts for multi-hop questions); if the sections cannot answer it,
    the answer is exactly 'not in policy'.
    """
    cited = _retrieve(question, k)
    parts = [f"[{i + 1}] {cid}\n{_chunk_text(cid)}" for i, cid in enumerate(cited)]
    context = "\n\n---\n\n".join(parts)
    if extra_context:
        context = "Additional structured facts:\n" + extra_context + "\n\n---\n\n" + context
    answer = _chat(LLM_ENDPOINT, SYSTEM_PROMPT, f"Context:\n{context}\n\nQuestion: {question}").strip()
    if re.fullmatch(r"not[ .]*in[ .]*policy[.!\s]*", answer, re.IGNORECASE):
        answer = NOT_IN_POLICY   # normalise to the exact phrase
    return {"answer": answer, "cited_chunk_ids": cited}


# smoke test: one answerable question and one unanswerable
for _q in ["How long is the initial waiting period on a Silver or Gold individual plan?",
           "What is the premium for adding a pet dog to a Saffron Shield plan?"]:
    _res = answer_policy_question(_q)
    print("Q:", _q)
    print("  cited:", _res["cited_chunk_ids"][:3], "...")
    print("  answer:", _res["answer"][:200].replace("\n", " "))


In [0]:
%python
# Step 2 — run the whole gold set plus U01/U02 through the chain and rebuild claims.genai.gold_answers.
# Spec: question_id STRING (gold-set ids, plus U01 and U02), question STRING, answer STRING,
#       cited_chunk_ids ARRAY<STRING> (top chunk ids in rank order, bare form, 1 to 5); one row per question.
# Overwrite confirmed by the user (the table previously held R4's retrieval-run output).

import json
import pandas as pd

GOLD_PATH = "/Volumes/claims/raw/landing/genai/gold_set.json"

with open(GOLD_PATH) as f:
    gold = json.load(f)

UNANSWERABLE = [
    ("U01", "What is the premium for adding a pet dog to a Saffron Shield plan?"),
    ("U02", "Does Saffron Shield pay for a member's gym membership?"),
]

# G19 is the multi-hop sentinel: it needs the claim's structured facts joined with the
# room-rent rules, so those facts are handed to the model alongside the retrieved sections.
_f = spark.sql("""
  SELECT plan_code_at_admission, plan_family, room_rent_per_day, room_rent_cap_per_day, room_rent_over_cap
  FROM claims.gold.claims_enriched WHERE claim_id = 'CLM-20003'
""").collect()[0]
G19_FACTS = (
    f"claim_id: CLM-20003; plan valid on admission: {_f.plan_code_at_admission} ({_f.plan_family}); "
    f"room rent per day: {_f.room_rent_per_day}; plan room rent cap per day: {_f.room_rent_cap_per_day}; "
    f"room rent over cap: {_f.room_rent_over_cap}."
)

rows = []
for q in gold["questions"]:
    qid, question = q["id"], q["question"]
    res = answer_policy_question(question, extra_context=G19_FACTS if qid == "G19" else None)
    rows.append({"question_id": qid, "question": question, **res})
    flag = "UNANSWERED" if res["answer"] == NOT_IN_POLICY else "answered"
    print(f"{qid}: {flag} | cited {len(res['cited_chunk_ids'])} | {res['answer'][:110].replace(chr(10), ' ')}")

for qid, question in UNANSWERABLE:
    res = answer_policy_question(question)
    rows.append({"question_id": qid, "question": question, **res})
    flag = "UNANSWERED" if res["answer"] == NOT_IN_POLICY else "ANSWERED (unexpected!)"
    print(f"{qid}: {flag} | cited {len(res['cited_chunk_ids'])} | {res['answer'][:110].replace(chr(10), ' ')}")

schema = "question_id STRING, question STRING, answer STRING, cited_chunk_ids ARRAY<STRING>"
answers_df = spark.createDataFrame(pd.DataFrame(rows), schema)
answers_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("claims.genai.gold_answers")

print("\nrows written:", spark.table("claims.genai.gold_answers").count())


In [0]:
%python
# Step 3 — verification against the spec.

tbl = spark.table("claims.genai.gold_answers")
rows = {r.question_id: r for r in tbl.collect()}

checks = []

# 1. shape: one row per question, exactly the gold ids + U01 + U02
expected_ids = {f"G{i:02d}" for i in range(1, 20)} | {"U01", "U02"}
checks.append(("21 rows, one per question (G01-G19 + U01 + U02)",
               tbl.count() == 21 and set(rows) == expected_ids))

# 2. cited_chunk_ids: 1..5 bare chunk ids (the policy_chunks id form '<doc>#<anchor>')
lens = [len(r.cited_chunk_ids) for r in rows.values()]
bare_ok = all(all("#" in c and c == c.strip() for c in r.cited_chunk_ids) for r in rows.values())
checks.append(("cited_chunk_ids: 1-5 bare chunk ids per row", all(1 <= n <= 5 for n in lens) and bare_ok))

# 3. unanswerables say exactly 'not in policy'
u_ok = all(rows[u].answer == "not in policy" for u in ("U01", "U02"))
checks.append(("U01 and U02 answer is exactly 'not in policy'", u_ok))

# 4. at least 16 of G01-G18 are answered, and none of them leak the phrase
single = [rows[f"G{i:02d}"] for i in range(1, 19)]
n_answered = sum(1 for r in single if r.answer != "not in policy")
n_leak = sum(1 for r in single if "not in policy" in r.answer.lower())
checks.append((">=16 of G01-G18 answered without the phrase leak", n_answered >= 16 and n_leak == 0))

# 5. answers are non-empty strings
checks.append(("every answer is a non-empty string", all(isinstance(r.answer, str) and r.answer.strip() for r in rows.values())))

for name, ok in checks:
    print(("PASS" if ok else "FAIL"), "-", name)
print(f"\nG01-G18 answered: {n_answered}/18; phrase leaks in single-doc questions: {n_leak}; U01/U02 -> 'not in policy': {u_ok}")
assert all(ok for _, ok in checks), "verification failed - see above"


In [0]:
-- Step 4 — the deliverable table: one row per question, answer + cited sections.
SELECT question_id, question, answer, cited_chunk_ids
FROM claims.genai.gold_answers
ORDER BY question_id;
